In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

# MandiMitra ~ Gemma Powered Crop Market Advisor

In [ ]:
!pip install -U "transformers>=5.9.0" accelerate --quiet

In [ ]:
# importing libraries
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
pd.set_option('display.max_columns', None)

## Step 1 - Loading the Dataset

In [ ]:
DATA_PATH = "/kaggle/input/datasets/anishaman07/agmarknet-india-commodity-prices-oct24-aug25/agmarknet-india-commodity-prices-2024-2025/agmarknet_india_historical_prices_2024_2025.csv"
# print(os.listdir(DATA_PATH))
df = pd.read_csv(DATA_PATH)


In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()


## Step 2 - Preprocessing

In [ ]:
# renaming the columns
# rename_map = {
#     "District Name": "district",
#     "Market Name": "market",
#     "Commodity": "commodity",
#     "Variety": "variety",
#     "Grade": "grade",
#     "Min Price (Rs./Quintal)": "min_price",
#     "Max Price (Rs./Quintal)": "max_price",
#     "Modal Price (Rs./Quintal)": "modal_price",
#     "Price Date": "price_date",
#     "State": "state",
# }
# df = df.rename(columns=rename_map)
# df = df.drop(columns=["Sl no."], errors="ignore")
# print(df.columns.tolist())

df.columns=['sl_no','district','market','commodity','variety','grade','min_price','max_price','modal_price','price_date','state']



In [ ]:
# matching the date format
df['price_date'] = pd.to_datetime(df['price_date'], format="%d %b %Y", errors="coerce")

mask = df['price_date'].isna()
if mask.any():
    df.loc[mask, 'price_date'] = pd.to_datetime(df.loc[mask, 'price_date'], errors="coerce", dayfirst=True)

print(f"Unparseable dates dropped: {df['price_date'].isna().sum()}")
df = df.dropna(subset=['price_date'])

In [ ]:
# dropping null, zero or negative prices
price_cols = ['min_price', 'max_price', 'modal_price']
df = df.dropna(subset=price_cols)
df = df[(df['modal_price'] > 0) & (df['min_price'] > 0) & (df['max_price'] > 0)]

# price should be between min and max (so dropping rows where it isn't)
df = df[(df['modal_price'] >= df['min_price']) & (df['modal_price'] <= df['max_price'])]

print(f"Rows after price cleaning: {len(df)}")

In [ ]:
# removing whitespace inconsistencies, for eg :- Wheat and wheat are same
for col in ['state', 'commodity', 'market', 'district', 'variety', 'grade']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

# drop exact duplicate rows (common in scraped Agmarknet data)
before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {before - len(df)}")
print(f"Rows after text normalization: {len(df)}")

In [ ]:
# Removing outlier prices per commodity using IQR
def remove_outliers(group, col='modal_price'):
    q1, q3 = group[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return group[(group[col] >= lower) & (group[col] <= upper)]

df = df.groupby('commodity', group_keys=False).apply(remove_outliers)
print(f"Rows after outlier removal: {len(df)}")

In [ ]:
# Only keeping commodity+state combos with enough data points for a meaningful trend
combo_counts = df.groupby(['commodity', 'state']).size()
valid_combos = combo_counts[combo_counts >= 5].index   # this dataset is small (wheat/UP only) so threshold is lower
df = df.set_index(['commodity', 'state']).loc[valid_combos].reset_index()

print(f"Final rows: {len(df)}")
print(f"Commodities available: {sorted(df['commodity'].unique())}")
print(f"States available: {sorted(df['state'].unique())}")
df.head()


In [ ]:
df.shape

# Aggregation Layer

In [ ]:
def get_market_summary(df, commodity, state, days=30):
    subset = df[(df['commodity'] == commodity) & (df['state'] == state)]
    subset = subset.sort_values('price_date')
    recent = subset.tail(days)

    if len(recent) < 2:
        return None

    first_price = recent.iloc[0]['modal_price']
    last_price = recent.iloc[-1]['modal_price']
    trend_pct = ((last_price - first_price) / first_price) * 100

    summary = {
        "commodity": commodity,
        "state": state,
        "date_range": f"{recent.iloc[0]['price_date'].date()} to {recent.iloc[-1]['price_date'].date()}",
        "current_price_per_quintal": round(last_price, 2),
        "avg_price": round(recent['modal_price'].mean(), 2),
        "min_price": round(recent['modal_price'].min(), 2),
        "max_price": round(recent['modal_price'].max(), 2),
        "trend_pct": round(trend_pct, 2),
        "num_data_points": len(recent),
    }
    return summary

example = get_market_summary(df, df['commodity'].iloc[0], df['state'].iloc[0])
example

In [ ]:
import transformers
print(transformers.__version__)


## WEATHER INTEGRATION

In [ ]:
STATE_COORDS = {
    "Uttar Pradesh": (26.8467, 80.9462),
    "Punjab": (31.1471, 75.3412),
    "Gujarat": (22.2587, 71.1924),
    "Maharashtra": (19.7515, 75.7139),
    "West Bengal": (22.9868, 87.8550),
    "Bihar": (25.0961, 85.3131),
    "Karnataka": (15.3173, 75.7139),
    "Tamil Nadu": (11.1271, 78.6569),
    "Rajasthan": (27.0238, 74.2179),
    "Madhya Pradesh": (22.9734, 78.6569),
    "Haryana": (29.0588, 76.0856),
    "Andhra Pradesh": (15.9129, 79.7400),
    "Telangana": (18.1124, 79.0193),
    "Kerala": (10.8505, 76.2711),
    "Odisha": (20.9517, 85.0985),
}
DEFAULT_COORD = (22.9734, 78.6569)  

In [ ]:
import requests

def get_weather_forecast(state, days=7):
    lat, lon = STATE_COORDS.get(state, DEFAULT_COORD)
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "precipitation_sum,temperature_2m_max,temperature_2m_min",
        "forecast_days": days,
        "timezone": "Asia/Kolkata",
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        d = resp.json()["daily"]
        total_rain = sum(d["precipitation_sum"])
        rainy_days = sum(1 for p in d["precipitation_sum"] if p > 1.0)
        avg_max_temp = sum(d["temperature_2m_max"]) / len(d["temperature_2m_max"])
        return {
            "forecast_days": days,
            "total_rain_mm": round(total_rain, 1),
            "rainy_days": rainy_days,
            "avg_max_temp_c": round(avg_max_temp, 1),
        }
    except Exception as e:
        print(f"Weather fetch failed: {e}")
        return None

# quick test
get_weather_forecast("Punjab")

# Sell-Now Vs. Wait-Now simulator

#### This integration is basically added to make the tool a decision-support calculator, not a describer of numbers that already existed(in chatbots). Given a farmer's actual quantity and a wait period, it computes a real rupee comparison

In [ ]:
PERISHABLE_STORAGE_COST = 8.0      # Rs per quintal per day — vegetables/fruits spoil fast
NON_PERISHABLE_STORAGE_COST = 1.5  # Rs per quintal per day — grains/pulses store cheaply

PERISHABLE_COMMODITIES = {
    "Tomato", "Onion", "Potato", "Brinjal", "Cauliflower", "Cabbage",
    "Apple", "Banana", "Mango", "Grapes", "Cucumber", "Carrot", "Peas",
}

def get_storage_cost_per_day(commodity):
    return PERISHABLE_STORAGE_COST if commodity in PERISHABLE_COMMODITIES else NON_PERISHABLE_STORAGE_COST


In [ ]:
def simulate_sell_vs_wait(summary, quantity_quintals, wait_days):
    current_price = summary['current_price_per_quintal']
    trend_pct = summary['trend_pct']

    # extrapolate the observed 30-day trend proportionally to the wait period
    projected_price_change_pct = trend_pct * (wait_days / 30)
    projected_price = current_price * (1 + projected_price_change_pct / 100)

    storage_cost_per_day = get_storage_cost_per_day(summary['commodity'])
    total_storage_cost = storage_cost_per_day * quantity_quintals * wait_days

    sell_now_value = current_price * quantity_quintals
    wait_value = (projected_price * quantity_quintals) - total_storage_cost
    difference = wait_value - sell_now_value

    return {
        "quantity_quintals": quantity_quintals,
        "wait_days": wait_days,
        "sell_now_value": round(sell_now_value, 2),
        "projected_price": round(projected_price, 2),
        "storage_cost_per_day": storage_cost_per_day,
        "total_storage_cost": round(total_storage_cost, 2),
        "wait_value": round(wait_value, 2),
        "difference": round(difference, 2),
        "better_choice": "Wait" if difference > 0 else "Sell Now",
    }

# quick test
simulate_sell_vs_wait(example, quantity_quintals=50, wait_days=10)


## loading the Gemma Model

In [ ]:
# !pip install -U git+https://github.com/huggingface/transformers.git --quiet

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

MODEL_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1"

processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch.float16,
    device_map="auto",
)

In [ ]:
def ask_gemma(prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]

    # enable_thinking=False keeps responses fast and to-the-point for this use case
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    # greedy decoding (do_sample=False) — faster than sampling, and gives consistent,
    # repeatable advice, which matters more than creative variety for financial guidance
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()

# quick sanity test — check how long this takes
import time
t0 = time.time()
print(ask_gemma("Say hello in one short sentence."))
print(f"\n(took {time.time() - t0:.1f}s)")

In [ ]:
def generate_insight(summary, weather=None, simulation=None, language="English"):
    if summary is None:
        return "Not enough recent data for this combination. Try another state or commodity."

    weather_block = ""
    if weather:
        weather_block = f"""
Weather forecast (next {weather['forecast_days']} days):
- Expected rainfall: {weather['total_rain_mm']} mm across {weather['rainy_days']} rainy day(s)
- Average max temperature: {weather['avg_max_temp_c']}°C
"""

    simulation_block = ""
    if simulation:
        simulation_block = f"""
Sell-Now vs. Wait calculation (already computed — do NOT recompute, just explain it):
- Quantity: {simulation['quantity_quintals']} quintals
- Value if sold today: Rs.{simulation['sell_now_value']}
- Value if held {simulation['wait_days']} days (projected price minus storage cost of Rs.{simulation['storage_cost_per_day']}/quintal/day): Rs.{simulation['wait_value']}
- Net difference (wait minus sell now): Rs.{simulation['difference']}
- Computed better choice: {simulation['better_choice']}
"""

    sections = ["1. Trend Summary (1 sentence)"]
    if weather:
        sections.append(f"{len(sections) + 1}. Weather Impact (1 sentence — how the forecast affects selling/storing this crop)")
    if simulation:
        sections.append(f"{len(sections) + 1}. Sell-Now vs Wait (1-2 sentences — restate the computed Rs. difference and which option is better, and why)")
    sections.append(f"{len(sections) + 1}. Recommendation (Sell now / Wait / Monitor — one-line reason, factoring in everything above)")
    sections.append(f"{len(sections) + 1}. One Risk to Watch (1 sentence)")
    sections_block = "\n".join(sections)

    prompt = f"""You are an agricultural market advisor helping Indian farmers.
Use ONLY the data below. Do not invent any numbers not given here — especially do not
recompute or alter the Sell-Now vs Wait numbers, just explain them in plain language.
Respond in {language}, in simple, plain language a farmer can understand.

Market Data:
- Commodity: {summary['commodity']}
- State: {summary['state']}
- Date range: {summary['date_range']}
- Current price: Rs.{summary['current_price_per_quintal']} per quintal
- 30-day average price: Rs.{summary['avg_price']}
- 30-day range: Rs.{summary['min_price']} - Rs.{summary['max_price']}
- Trend: {summary['trend_pct']}% change over this period
{weather_block}{simulation_block}
Give exactly these short sections:
{sections_block}
"""
    return ask_gemma(prompt)

In [ ]:
weather_example = get_weather_forecast(example['state'])
print(generate_insight(example, weather=weather_example))

In [ ]:
def get_confidence(summary):
    n = summary['num_data_points']
    if n >= 20:
        return "🟢 High Confidence", "Based on strong recent data"
    elif n >= 10:
        return "🟡 Medium Confidence", "Based on limited recent data"
    else:
        return "🔴 Low Confidence", "Very few data points — treat as indicative only"

In [ ]:
!pip install plotly --quiet

In [ ]:
import gradio as gr
import plotly.graph_objects as go
import traceback

commodities = sorted(df['commodity'].unique().tolist())
states = sorted(df['state'].unique().tolist())

def make_price_chart(commodity, state, days=30):
    subset = df[(df['commodity'] == commodity) & (df['state'] == state)].sort_values('price_date').tail(days)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=subset['price_date'], y=subset['modal_price'], mode='lines+markers',
                              name='Modal Price', line=dict(color='#2E8B57', width=3), marker=dict(size=6)))
    fig.add_trace(go.Scatter(x=subset['price_date'], y=subset['min_price'], mode='lines',
                              name='Min Price', line=dict(color='#90EE90', width=1, dash='dot')))
    fig.add_trace(go.Scatter(x=subset['price_date'], y=subset['max_price'], mode='lines',
                              name='Max Price', line=dict(color='#FF6347', width=1, dash='dot')))
    fig.update_layout(title=f"{commodity} Price Trend — {state}", xaxis_title="Date",
                       yaxis_title="Price (Rs./Quintal)", template="plotly_white",
                       height=350, margin=dict(l=40, r=20, t=50, b=40))
    return fig

def make_simulation_chart(simulation):
    winner = simulation['better_choice']
    colors = ['#2E8B57' if winner == 'Sell Now' else '#4682B4',
              '#2E8B57' if winner == 'Wait' else '#4682B4']
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=["Sell Now", f"Wait {simulation['wait_days']} days"],
        y=[simulation['sell_now_value'], simulation['wait_value']],
        marker_color=colors,
        text=[f"₹{simulation['sell_now_value']:,.0f}", f"₹{simulation['wait_value']:,.0f}"],
        textposition='outside',
    ))
    fig.update_layout(
        title=f"Sell Now vs Wait — {simulation['quantity_quintals']} quintals",
        yaxis_title="Total Value (Rs.)", template="plotly_white",
        height=350, showlegend=False, margin=dict(l=40, r=20, t=50, b=60),
    )
    return fig

def _app_fn_impl(commodity, state, language, include_weather, quantity, wait_days):
    summary = get_market_summary(df, commodity, state)
    if summary is None:
        return "Not enough recent data for this combination.", None, "", None, "", None

    weather = get_weather_forecast(state) if include_weather else None
    simulation = simulate_sell_vs_wait(summary, quantity_quintals=quantity, wait_days=wait_days)

    confidence, note = get_confidence(summary)
    insight = generate_insight(summary, weather=weather, simulation=simulation, language=language)
    insight_with_badge = f"{confidence} — {note}\n\n{insight}"

    weather_row = ""
    if weather:
        weather_row = f"| 7-day Rain Forecast | {weather['total_rain_mm']} mm ({weather['rainy_days']} rainy days) |\n| Avg Max Temp | {weather['avg_max_temp_c']}°C |\n"

    stat_cards = f"""
### 📊 Quick Stats
| Metric | Value |
|---|---|
| Current Price | ₹{summary['current_price_per_quintal']}/quintal |
| 30-day Average | ₹{summary['avg_price']}/quintal |
| 30-day Range | ₹{summary['min_price']} – ₹{summary['max_price']} |
| Trend | {summary['trend_pct']}% |
{weather_row}| Data Points | {summary['num_data_points']} |
"""

    price_chart = make_price_chart(commodity, state)
    sim_chart = make_simulation_chart(simulation)

    winner_emoji = "🟢" if simulation['better_choice'] == "Wait" else "🔵"
    sim_summary = f"""
### ⚖️ Sell-Now vs Wait ({quantity} quintals, {wait_days} days)
| Option | Total Value |
|---|---|
| Sell Now | ₹{simulation['sell_now_value']:,.0f} |
| Wait {wait_days} days | ₹{simulation['wait_value']:,.0f} |
| **Difference** | **₹{simulation['difference']:,.0f}** |

{winner_emoji} **Computed better choice: {simulation['better_choice']}**
(assumes storage cost of ₹{simulation['storage_cost_per_day']}/quintal/day — see Limitations in writeup)
"""

    summary_df = pd.DataFrame([summary])
    return insight_with_badge, price_chart, stat_cards, sim_chart, sim_summary, summary_df

def app_fn(commodity, state, language, include_weather, quantity, wait_days):
    # Wraps _app_fn_impl so ANY error shows up directly in the Advisory box
    # instead of a generic "Error" toast with no detail.
    try:
        return _app_fn_impl(commodity, state, language, include_weather, quantity, wait_days)
    except Exception:
        err_text = traceback.format_exc()
        error_message = f"⚠️ Something went wrong — full error below:\n\n{err_text}"
        return error_message, None, "", None, "", None

custom_css = """
.gradio-container {max-width: 1150px !important}
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="green", secondary_hue="orange"), css=custom_css) as demo:
    gr.Markdown("# 🌾 MandiMitra — Crop Market Advisor")
    gr.Markdown("Gemma-powered advisory combining **real mandi prices**, **live weather**, and a **computed sell-vs-wait simulator** — grounded, not guessed.")

    with gr.Row():
        with gr.Column(scale=1):
            commodity_in = gr.Dropdown(commodities, label="Commodity", value=commodities[0])
            state_in = gr.Dropdown(states, label="State", value=states[0])
            lang_in = gr.Dropdown(["English", "Hindi", "Bengali", "Tamil", "Telugu"], value="English", label="Response Language")
            weather_toggle = gr.Checkbox(label="🌦️ Include weather forecast", value=True)
            gr.Markdown("**Your Holding**")
            quantity_in = gr.Number(label="Quantity (quintals)", value=50, minimum=1)
            wait_days_in = gr.Slider(label="Days you could wait", minimum=1, maximum=30, value=10, step=1)
            submit_btn = gr.Button("🔎 Get Advisory", variant="primary")

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("📋 Advisory"):
                    advisory_out = gr.Textbox(label="Advisory", lines=9)
                    stats_out = gr.Markdown()
                with gr.Tab("📈 Price Trend"):
                    chart_out = gr.Plot(label="Price Trend (Last 30 Days)")
                with gr.Tab("⚖️ Sell vs Wait"):
                    sim_chart_out = gr.Plot(label="Sell Now vs Wait — Computed Comparison")
                    sim_summary_out = gr.Markdown()
                with gr.Tab("🔍 Raw Data"):
                    data_out = gr.Dataframe(label="Underlying Data (verify it yourself)")

    submit_btn.click(fn=app_fn,
                      inputs=[commodity_in, state_in, lang_in, weather_toggle, quantity_in, wait_days_in],
                      outputs=[advisory_out, chart_out, stats_out, sim_chart_out, sim_summary_out, data_out])

demo.launch(share=True)